# Kickstarter project success



## The Machine Learning Workflow

### Define the Business Goal

In recent years, the range of funding options for projects created by individuals and small companies has expanded considerably. Alongside savings, bank loans, and friends and family funding, crowdfunding has become a popular and readily available alternative.

Kickstarter, founded in 2009, is a well-known crowdfunding platform. It uses an all-or-nothing funding model: a project is only funded if it meets its goal amount, otherwise backers give no money to the project.

Many factors contribute to the success or failure of a project, in general and also on Kickstarter. Some of these can be quantified or categorised, which makes it possible to build a model that predicts whether a project will succeed. The aim of this project is to build such a model and to analyse Kickstarter project data more generally, so you can help potential creators assess whether Kickstarter is a good funding option for them and what their chances of success are.

### Get the Data

We load the cleaned Arabica dataset. It holds both the coffee attributes and the overall cupping score (`Total.Cup.Points`), which we will use as the target.

In [2]:
import pandas as pd 

kickstarter = pd.read_csv("data/kickstarter_preprocessed.csv/kickstarter_preprocessed.csv", encoding="windows-1252")

Let us preview the data to see what we are working with.

### First Look at the Data

A quick structural check shows the column types and where values are missing.

In [3]:
kickstarter.head()

,state,usd_goal_real,name_length,duration_days,launch_year,launch_month,launch_day,country_AU,country_CA,country_DE,...,category_Software,category_Tabletop Games,category_Technology,category_Theater,category_Video Games,category_Wearables,category_Web,category_Webseries,category_Woodworking,category_World Music
0,0,1533.95,31,51,2015,8,11,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,30000.00,45,51,2017,9,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,43961.63,14,45,2013,1,12,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,5000.00,49,30,2012,3,17,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,19500.00,58,51,2015,7,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
kickstarter.info()

<class 'pandas.DataFrame'>
RangeIndex: 372293 entries, 0 to 372292
Data columns (total 92 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   state                      372293 non-null  int64  
 1   usd_goal_real              372293 non-null  float64
 2   name_length                372293 non-null  int64  
 3   duration_days              372293 non-null  int64  
 4   launch_year                372293 non-null  int64  
 5   launch_month               372293 non-null  int64  
 6   launch_day                 372293 non-null  int64  
 7   country_AU                 372293 non-null  int64  
 8   country_CA                 372293 non-null  int64  
 9   country_DE                 372293 non-null  int64  
 10  country_ES                 372293 non-null  int64  
 11  country_FR                 372293 non-null  int64  
 12  country_GB                 372293 non-null  int64  
 13  country_IT                 372293 non-nu

The dataset has 323,750 entries, and 11 usable variables (category, main category, type of curreny used, deadline, goal, launched date, pledged, state, backers, country,usd pleged)

Next step; remove unnecessary variables from the original dataset

## Clean the Data

These steps are all **leakage-safe**, so we do them before the split: tidy the column names, drop columns and rows we cannot use, fix obvious data-entry errors, and remove features that would give away the answer. The only step we leave for after the split is imputation, because its fill values are learned from the data.

In [74]:
# Remove columns that are not needed
kickstarter.columns = kickstarter.columns.str.strip()
columns_to_remove = ["ID", "name", "Unnamed: 13", "Unnamed: 14", "Unnamed: 15","Unnamed: 16" ]
kickstarter = kickstarter.drop(columns=columns_to_remove, errors="ignore")

# Clean the remaining column names
kickstarter.columns = (
    kickstarter.columns.str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)
kickstarter.columns.tolist()

['category',
 'main_category',
 'currency',
 'deadline',
 'goal',
 'launched',
 'pledged',
 'state',
 'backers',
 'country',
 'usd_pledged']

Fix the variable types so that they reflect the variables nature.

In [75]:

kickstarter.shape 

(323750, 11)

In [76]:
kickstarter.isna().sum()

category            5
main_category       0
currency            0
deadline            0
goal                0
launched            0
pledged             0
state               0
backers             0
country             0
usd_pledged      3790
dtype: int64

In [80]:
date_columns = ["deadline", "launched"]
num_columns = ["pledged", "usd_pledged", "goal", "backers"]

for column in date_columns:
    kickstarter[column] = pd.to_datetime(kickstarter[column], format = "mixed")

for column in num_columns:
    kickstarter[column] = pd.to_numeric(kickstarter[column])

print(kickstarter[date_columns].dtypes, kickstarter[num_columns].dtypes) 

DateParseError: Unknown datetime string format, unable to parse: SEK

In [27]:
# checking data types conversions.
kickstarter.isna().sum()    

category            5
main_category       0
currency            0
deadline          632
goal              632
launched          624
pledged           624
state               0
backers           623
country             0
usd_pledged      4413
dtype: int64

In [ ]:
## Handling missing values

In [8]:
# Drop columns that are missing more than half of their values
missing_fraction = kickstarter.isna().mean().sort_values(ascending=False)
high_missing = missing_fraction[missing_fraction > 0.5].index
print("Dropping columns missing more than 50% of values:", list(high_missing))
kickstarter = kickstarter.drop(columns=high_missing)

# Add the exact column names you want to remove
columns_to_remove = []  # Example: ["Owner", "Farm.Name"]
kickstarter = kickstarter.drop(columns=columns_to_remove)

Dropping columns missing more than 50% of values: []


Only wo variables are missing`category`, only 5 rows and `usd_pledged`, 3790 rows, since there is no is missing more than half of its values, so we do not drop any variables. As for the missing variables, I suggest imputation techniqus; for `category`, mode since it is categorical in nature and mean for `usd_pledged`, since it is numeric in nature.

In [10]:
# Imputing missing values for categorical columns with the mode and numeric with mean

kickstarter["category"] = kickstarter["category"].fillna(kickstarter["category"].mode()[0]) 
kickstarter["usd_pledged"] = kickstarter["usd_pledged"].fillna(kickstarter["usd_pledged"].mean())


In [11]:
# checking there is no missing values in the dataset
kickstarter.isna().sum()    

category           0
main_category      0
currency           0
deadline         632
goal             632
launched         624
pledged          624
state              0
backers          623
country            0
usd_pledged        0
dtype: int64

The component scores add up to the total (correlation 1.00). That makes them **target leakage**: feeding them to the model would just hand it the answer broken into ten pieces. They are also unrealistic for our goal, because the buyer wants an estimate *before* tasting, when these cupping scores do not exist yet.

So we drop the component scores now and predict the quality from genuinely independent attributes instead.

In [ ]:
# The target is the overall cupping score
y = coffee["total_cup_points"]

# Features: numeric columns, minus the row index, the target, and the leaky component scores
X = coffee.select_dtypes("number").drop(
    columns=["unnamed_0", "total_cup_points"] + component_scores
)
X.columns.tolist()

This leaves eight independent attributes: the number of bags, moisture, the two defect counts, quakers, and the three altitude columns. These are things a buyer could know about a lot without cupping it.

## Train-Test Split

We hold out part of the data so we can judge the model on coffees it has never seen. From here on, anything we *fit* (such as imputation values) is learned on the training set only.

We use a single train-test split (80% train, 20% test). A larger project would also keep a validation set, or use cross validation, to tune the model without touching the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

The 1310 coffees are split into 1048 for training and 262 for testing.

## Explore the Training Data

Now we explore the **training set** to understand the features before fitting anything.

In [ ]:
X_train.info()

There are eight numeric features. Most are complete, but `quakers` has one missing value and the three altitude columns are missing for 187 of the 1048 training coffees. We will impute those gaps after the split.

In [ ]:
# How does each remaining feature correlate with the target?
X_train.corrwith(y_train).sort_values(ascending=False)

None of the independent attributes is strongly related to the score. The clearest signal is `category_two_defects` (about -0.28: more defects, lower score), followed by moisture and altitude, all weak.

## Feature Engineering

We fill every missing value (`quakers` and the three altitude columns) with the **training** mean of its column, then reuse those values on the test set.

In [ ]:
# Learn the fill values on the training set only
train_means = X_train.mean()
X_train = X_train.fillna(train_means)
train_means

`train_means` holds one value per column. We reuse this exact set of values on the test set at evaluation time, which is what keeps the evaluation honest.

## Baseline Model

Before training a model, we set a **baseline**: the simplest rule that makes a prediction.

For a regression like ours, a standard heuristic is to ignore the features and always predict the **average training score**. If our model cannot beat that, it has learned nothing useful.

In [ ]:
# Heuristic baseline: ignore the features, always predict the mean training score
from sklearn.metrics import root_mean_squared_error

baseline_prediction = y_train.mean()
baseline_rmse = root_mean_squared_error(y_test, [baseline_prediction] * len(y_test))

print(f"Baseline always predicts: {baseline_prediction:.2f}")
print(f"Baseline test RMSE: {baseline_rmse:.2f}")

The baseline always predicts about 82.2 (the mean training score) and reaches a test RMSE of about 2.66, so on average it misses the true score by roughly 2 to 3 points. This is the number our model has to beat.

## Train the Model

We train a linear regression model: fast, easy to read, and a sensible step up from the baseline. First we set the test set aside and save a copy, then we fit the model and check the error on the training data.

In [ ]:
# Set the test set aside and save a copy. We do not touch it until the final evaluation.
X_test.to_csv("data/X_test.csv")
y_test.to_csv("data/y_test.csv")

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
y_train_pred = reg.predict(X_train)
rmse = root_mean_squared_error(y_train, y_train_pred)
print(rmse)

The training RMSE is about 2.54 points, close to the baseline. The model is not memorising the data, which is unsurprising given how weakly the features relate to the score.

## Evaluate the Model

We apply the same fitted step to the test set: imputation with the **training** means (never recomputed from the test data). The cleaning and the altitude fix already happened before the split, so the test set carries them too. Then we score the model on these unseen coffees.

In [ ]:
# Apply the fitted step (imputation) to the test set, using the training means
X_test = X_test.fillna(train_means)

In [ ]:
y_test_pred = reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_test_pred)
print(rmse)

The test RMSE is about 2.46, just below the baseline's 2.66. The model is only a little better than always guessing the average: the attributes a buyer can know before tasting (altitude, defects, moisture, bag count) carry only weak signal about the final cupping score.

The natural next steps in the workflow,  are trying richer features or models, selecting the best one on validation data or cross-validation, hyperparameter tuning, doing error analysis, reporting a final test score, then deploying and monitoring the model.

In [ ]:
# save the best model to a file for later use
import joblib
joblib.dump(reg, "models/linear_regression_model.joblib")

In [ ]:
# use the model to predict the score of a new coffee with these attributes:
 
new_coffee = pd.DataFrame({
    
    "number_of_bags" :[1] ,
    "moisture": [10.5] , 
    "category_one_defects": [0],
    "quakers": [0],
    "category_two_defects": [1], 
    "altitude_low_meters": [1200], 
    "altitude_high_meters": [1500],
    "altitude_mean_meters": [1350]
})
new_coffee = new_coffee.fillna(train_means)  # apply the same imputation as training data

In [ ]:
X_train.columns

In [ ]:
# load the model from the file and use it to predict the score of the new coffee
loaded_model = joblib.load("models/linear_regression_model.joblib")
predicted_score = loaded_model.predict(new_coffee)[0]
print(f"Predicted cupping score for the new coffee: {predicted_score:.2f}")

## References & Further Reading

- [**LinearRegression (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html): the model trained in this notebook.
- [**train_test_split (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html): how the held-out test set is created.
- [**root_mean_squared_error (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): the metric used to evaluate the model and the baseline.
- [**Common pitfalls and recommended practices (scikit-learn)**](https://scikit-learn.org/stable/common_pitfalls.html): how data leakage happens and how to avoid it.
- [**Coffee Quality Database**](https://github.com/jldbc/coffee-quality-database): the source of the dataset used here.